# Phase 5i: Artist-filtered evaluation

Reviewer concern: the 2,075-track catalog has only ~1,086 artists, and same-artist tracks share a quadrant far more often
than chance. Retrieval searches the whole catalog, so same-artist neighbours may inflate P@10 (the album/artist effect),
possibly more for audio than for lyrics.

This notebook re-runs the Phase 5f main results with **every same-artist candidate removed** from each query's ranking
(and from the ideal ranking for nDCG). It reports:
1. how often unfiltered top-10 lists contain the query's artist, per strategy;
2. the main table and tests under artist filtering (configurations from Phase 5f, re-selected on validation as a check);
3. the paired change (filtered − unfiltered) per strategy;
4. TOST for audio vs lyrics, per-quadrant P@10, and artist-cluster bootstrap CIs.

Outputs are new files (`artist_*.csv`); nothing from earlier phases is overwritten.

In [1]:
import os, re, unicodedata
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon, t as t_dist
from sklearn.preprocessing import StandardScaler

ROOT = os.environ.get('MUSIC_RESEARCH_ROOT', os.path.abspath('.'))
OUT_DIR = os.path.join(ROOT, 'outputs')
ENC_DIR = os.path.join(OUT_DIR, 'encoders')
SIGMA, KMAX, N_BOOT, SEED = 0.10, 20, 10_000, 42
ALPHAS  = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
SCHEMES = {'tvt_70_15_15': '70/15/15', 'tvt_40_30_30': '40/30/30'}
QUADS   = ['Q1', 'Q2', 'Q3', 'Q4']
rng = np.random.default_rng(SEED)

lyrics_df = pd.read_parquet(os.path.join(OUT_DIR, 'lyrics_features.parquet'))
merged    = pd.read_parquet(os.path.join(OUT_DIR, 'merged_dataset.parquet'))
cat_sids  = lyrics_df['song_id'].tolist()
cat_pos   = {s: i for i, s in enumerate(cat_sids)}
a_ids = pd.read_csv(os.path.join(ENC_DIR, 'audio_song_ids.csv'))['song_id'].tolist()
a_row = [{s: i for i, s in enumerate(a_ids)}[s] for s in cat_sids]
assert pd.read_csv(os.path.join(ENC_DIR, 'text_song_ids.csv'))['song_id'].tolist() == cat_sids

muq  = np.load(os.path.join(ENC_DIR, 'audio_muq.npy'),  mmap_mode='r')[a_row].astype(np.float64)
mert = np.load(os.path.join(ENC_DIR, 'audio_mert.npy'), mmap_mode='r')[a_row].astype(np.float64)
a_old = pd.read_parquet(os.path.join(OUT_DIR, 'audio_features.parquet')).set_index('song_id').loc[cat_sids]
REPS = {'sbert': np.load(os.path.join(ENC_DIR, 'text_sbert_mpnet_trunc.npy')).astype(np.float64),
        'clap':  np.vstack(a_old['audio_clap'].values).astype(np.float64)}
for L in range(13):
    REPS[f'muq_L{L}'] = muq[:, L]; REPS[f'mert_L{L}'] = mert[:, L]
del muq, mert
FAMILY = lambda r: r.split('_L')[0]

lab  = merged.set_index('song_id').loc[cat_sids]
quad = lab['quadrant'].values
Y    = lab[['arousal', 'valence']].values.astype(np.float64)
D_AV = np.linalg.norm(Y[:, None] - Y[None], axis=-1)

mm = merged[merged['is_multimodal']]
lyr_stem_to_sid = {os.path.splitext(os.path.basename(p))[0]: s for s, p in zip(mm['song_id'], mm['lyrics_path'])}
def read_split(modality, scheme, subset):
    folder = 'MERGE_Audio_Balanced' if modality == 'audio' else 'MERGE_Lyrics_Balanced'
    ids = pd.read_csv(os.path.join(ROOT, folder, 'tvt_dataframes', scheme,
                                   f'{scheme}_{subset}_{modality}_balanced.csv'))['Song']
    return set(ids.map(lyr_stem_to_sid).dropna()) if modality == 'lyrics' else set(ids)
cat = set(cat_sids)
shared = {lbl: {sub: sorted(read_split('audio', sc, sub) & read_split('lyrics', sc, sub) & cat)
                for sub in ['train', 'validate', 'test']} for sc, lbl in SCHEMES.items()}
print({lbl: {k: len(v) for k, v in d.items()} for lbl, d in shared.items()})

{'70/15/15': {'train': 1187, 'validate': 305, 'test': 302}, '40/30/30': {'train': 475, 'validate': 616, 'test': 617}}


## Artist groups

Primary grouping: artist name after Unicode folding, lower-casing, dropping a leading "The" and punctuation
(this merges "The Prodigy" / "Prodigy"). Sensitivity grouping ("leader"): additionally keep only the part before
"&", "and", "feat.", "with", so "Bob Marley & the Wailers" and "Bob Marley" would fall together. All merges made by the
looser rule are printed for inspection.

In [2]:
def fold(s):
    s = unicodedata.normalize('NFKD', str(s)).encode('ascii', 'ignore').decode().lower().strip()
    return re.sub(r'^the\s+', '', s)
def norm(s):
    return re.sub(r'[^a-z0-9]+', ' ', fold(s)).strip()
def leader(s):
    s = re.split(r'\s*(?:&|\+|/|\band\b|\bfeat\b\.?|\bfeaturing\b|\bft\b\.?|\bwith\b|\bvs\b\.?)\s*', fold(s))[0]
    return re.sub(r'[^a-z0-9]+', ' ', s).strip() or norm(s)

artists = lab['artist'].astype(str).values
GROUPS = {'artist': np.array([norm(a) for a in artists]), 'leader': np.array([leader(a) for a in artists])}
for g, v in GROUPS.items():
    print(f'{g}: {len(set(v))} groups')
chk = pd.DataFrame({'raw': artists, 'artist': GROUPS['artist'], 'leader': GROUPS['leader']}).drop_duplicates('raw')
extra = chk[chk.duplicated('leader', keep=False) & ~chk.duplicated('artist', keep=False)].sort_values('leader')
print('Merges added by the leader rule:'); print(extra.to_string(index=False))

SAME = {g: v[:, None] == v[None] for g, v in GROUPS.items()}
for g in SAME:
    np.fill_diagonal(SAME[g], False)

art = GROUPS['artist']; cnt = pd.Series(art).value_counts()
n_other = SAME['artist'].sum(1)
same_q = [(quad[SAME['artist'][i]] == quad[i]).mean() for i in range(len(quad)) if n_other[i] > 0]
print(f'tracks with >=1 same-artist track: {(n_other > 0).sum()} / {len(quad)};  mean same-artist candidates per query: {n_other.mean():.2f}')
print(f'P(same quadrant | same artist) = {np.mean(same_q):.3f}  (P(same quadrant | random other track) = '
      f'{np.mean([(quad[np.arange(len(quad)) != i] == quad[i]).mean() for i in range(len(quad))]):.3f})')
print('Largest artists:', cnt.head(8).to_dict())

artist: 1086 groups
leader: 1082 groups
Merges added by the leader rule:
                             raw                         artist       leader
        Bob Marley & the Wailers         bob marley the wailers   bob marley
                      Bob Marley                     bob marley   bob marley
  Hank Ballard & the Midnighters   hank ballard the midnighters hank ballard
                    Hank Ballard                   hank ballard hank ballard
                      Huey Lewis                     huey lewis   huey lewis
           Huey Lewis & the News            huey lewis the news   huey lewis
                    Ziggy Marley                   ziggy marley ziggy marley
Ziggy Marley & the Melody Makers ziggy marley the melody makers ziggy marley


tracks with >=1 same-artist track: 1376 / 2075;  mean same-artist candidates per query: 3.48
P(same quadrant | same artist) = 0.735  (P(same quadrant | random other track) = 0.251)
Largest artists: {'megadeth': 27, 'slayer': 25, 'billie holiday': 24, 'ella fitzgerald': 20, 'bob dylan': 17, 'jimmy reed': 16, 'elvis presley': 15, 'beatles': 14}


## Retrieval machinery (identical to Phase 5f)

In [3]:
_unit = {}
def unit(rep, how):
    if (rep, how) not in _unit:
        X = REPS[rep]; X = StandardScaler().fit_transform(X) if how == 'z' else X
        _unit[(rep, how)] = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-12)
    return _unit[(rep, how)]

def score_matrix(cfg, rows):
    if cfg[0] == 'single':
        X = unit(*cfg[1]); return X[rows] @ X.T
    A, T = unit(*cfg[1]), unit(*cfg[2])
    if cfg[0] == 'late':
        return cfg[3] * (A[rows] @ A.T) + (1 - cfg[3]) * (T[rows] @ T.T)
    E = np.hstack([A, T]) / np.sqrt(2)
    return E[rows] @ E.T

DISC = 1 / np.log2(np.arange(2, KMAX + 2))
def topk(S, k):
    top = np.argpartition(-S, k, 1)[:, :k]
    return np.take_along_axis(top, np.take_along_axis(S, top, 1).argsort(1)[:, ::-1], 1)

def evaluate(cfg, qs, exclude=None):
    rows = np.array([cat_pos[s] for s in qs])
    S = score_matrix(cfg, rows); S[np.arange(len(rows)), rows] = -np.inf
    if exclude is not None:
        S[exclude[rows]] = -np.inf
    PK = np.column_stack([(quad[topk(S, k)] == quad[rows][:, None]).mean(1) for k in range(1, KMAX + 1)])
    top = topk(S, 10)
    d = np.take_along_axis(D_AV[rows], top, 1)
    Dq = D_AV[rows].copy(); Dq[np.arange(len(rows)), rows] = np.inf
    if exclude is not None:
        Dq[exclude[rows]] = np.inf
    g = np.exp(-d**2 / (2 * SIGMA**2)); gi = np.exp(-np.sort(Dq, 1)[:, :10]**2 / (2 * SIGMA**2))
    return {'P@K': PK, 'nDCG@10': (g * DISC[:10]).sum(1) / (gi * DISC[:10]).sum(1),
            'AVdist@10': d.mean(1), 'quadrant': quad[rows], 'song_id': np.array(qs), 'top10': top, 'rows': rows}

def best_single(lbl, family, exclude=None):
    cands = [(r, h) for r in REPS if FAMILY(r) == family for h in ['raw', 'z']]
    return max(cands, key=lambda c: evaluate(('single', c), shared[lbl]['validate'], exclude)['P@K'][:, 9].mean())

def best_alpha(lbl, ca, ct, exclude=None):
    return max(ALPHAS, key=lambda al: evaluate(('late', ca, ct, al), shared[lbl]['validate'], exclude)['P@K'][:, 9].mean())

def build_cfg(exclude=None):
    CFG = {}
    for lbl in SCHEMES.values():
        c = {f: best_single(lbl, f, exclude) for f in ['muq', 'mert', 'sbert', 'clap']}
        a_muq, a_clap = best_alpha(lbl, c['muq'], c['sbert'], exclude), best_alpha(lbl, c['clap'], c['sbert'], exclude)
        CFG[lbl] = {'audio-only (MuQ)': ('single', c['muq']),
                    'audio-only (MERT)': ('single', c['mert']),
                    'lyrics-only (SBERT)': ('single', c['sbert']),
                    'late fusion MuQ+SBERT': ('late', c['muq'], c['sbert'], a_muq),
                    'early fusion (≡ α=0.5)': ('early', c['muq'], c['sbert']),
                    'audio-only (CLAP)': ('single', c['clap']),
                    'late fusion CLAP+SBERT': ('late', c['clap'], c['sbert'], a_clap)}
    return CFG

desc = lambda cfg: ' + '.join(f'{r}/{h}' for r, h in cfg[1:3] if isinstance(r, str)) + (f' α={cfg[3]}' if cfg[0] == 'late' else '')
CFG_UNF = build_cfg(None)                    # Phase 5f selection (unfiltered validation)
CFG_ART = build_cfg(SAME['artist'])          # re-selected on artist-filtered validation
sel = pd.DataFrame([{'split': l, 'strategy': s, 'unfiltered selection': desc(CFG_UNF[l][s]),
                     'artist-filtered selection': desc(CFG_ART[l][s])} for l in SCHEMES.values() for s in CFG_UNF[l]])
sel['changed'] = sel['unfiltered selection'] != sel['artist-filtered selection']
print(sel.to_string(index=False))
sel.to_csv(os.path.join(OUT_DIR, 'artist_filtered_config_selection.csv'), index=False)

   split               strategy     unfiltered selection artist-filtered selection  changed
70/15/15       audio-only (MuQ)                 muq_L6/z                  muq_L6/z    False
70/15/15      audio-only (MERT)                mert_L7/z                 mert_L7/z    False
70/15/15    lyrics-only (SBERT)                  sbert/z                   sbert/z    False
70/15/15  late fusion MuQ+SBERT muq_L6/z + sbert/z α=0.6  muq_L6/z + sbert/z α=0.6    False
70/15/15 early fusion (≡ α=0.5)       muq_L6/z + sbert/z        muq_L6/z + sbert/z    False
70/15/15      audio-only (CLAP)                   clap/z                    clap/z    False
70/15/15 late fusion CLAP+SBERT   clap/z + sbert/z α=0.2    clap/z + sbert/z α=0.1     True
40/30/30       audio-only (MuQ)                 muq_L6/z                  muq_L6/z    False
40/30/30      audio-only (MERT)                mert_L6/z                 mert_L6/z    False
40/30/30    lyrics-only (SBERT)                  sbert/z                   sbert

## 1. Same-artist neighbours in the unfiltered top-10

`same_artist@10` is the share of the top-10 by the query's artist; `chance` is that share under a uniformly random
ranking (same-artist candidates / 2,074). `any_same_artist@10` is the share of queries with at least one.

In [4]:
RES_UNF = {l: {s: evaluate(c, shared[l]['test']) for s, c in CFG_UNF[l].items()} for l in SCHEMES.values()}

def boot_ci(x, n=N_BOOT):
    idx = rng.integers(0, len(x), (n, len(x)))
    return np.percentile(x[idx].mean(1), [2.5, 97.5])

orows = []
for l, res in RES_UNF.items():
    for s, m in res.items():
        hit = SAME['artist'][m['rows'][:, None], m['top10']]           # (n, 10) same-artist indicator
        frac = hit.mean(1)
        sq = quad[m['top10']] == m['quadrant'][:, None]
        lo, hi = boot_ci(frac)
        orows.append({'split': l, 'strategy': s, 'same_artist@10': frac.mean(), 'lo': lo, 'hi': hi,
                      'any_same_artist@10': (hit.sum(1) > 0).mean(),
                      'chance': (SAME['artist'][m['rows']].sum(1) / (len(cat_sids) - 1)).mean(),
                      'quad_precision_same_artist_hits': sq[hit].mean() if hit.any() else np.nan,
                      'quad_precision_other_hits': sq[~hit].mean(),
                      'share_of_correct_hits_from_same_artist': (sq & hit).sum() / sq.sum()})
overlap = pd.DataFrame(orows)
print(overlap.to_string(index=False, float_format=lambda x: f'{x:.3f}'))
overlap.to_csv(os.path.join(OUT_DIR, 'artist_overlap_top10.csv'), index=False)

   split               strategy  same_artist@10    lo    hi  any_same_artist@10  chance  quad_precision_same_artist_hits  quad_precision_other_hits  share_of_correct_hits_from_same_artist
70/15/15       audio-only (MuQ)           0.077 0.063 0.092               0.391   0.002                            0.892                      0.603                                   0.110
70/15/15      audio-only (MERT)           0.066 0.054 0.079               0.358   0.002                            0.870                      0.600                                   0.093
70/15/15    lyrics-only (SBERT)           0.032 0.024 0.042               0.185   0.002                            0.938                      0.594                                   0.050
70/15/15  late fusion MuQ+SBERT           0.081 0.066 0.097               0.391   0.002                            0.902                      0.684                                   0.104
70/15/15 early fusion (≡ α=0.5)           0.073 0.059 0.088 

## 2. Main table, unfiltered vs artist-filtered (Phase 5f configurations)

In [5]:
RES_ART  = {l: {s: evaluate(c, shared[l]['test'], SAME['artist']) for s, c in CFG_UNF[l].items()} for l in SCHEMES.values()}
RES_LEAD = {l: {s: evaluate(c, shared[l]['test'], SAME['leader']) for s, c in CFG_UNF[l].items()} for l in SCHEMES.values()}
RES_RESEL = {l: {s: evaluate(c, shared[l]['test'], SAME['artist']) for s, c in CFG_ART[l].items()} for l in SCHEMES.values()}

# sanity: unfiltered numbers reproduce Phase 5f
ref = pd.read_csv(os.path.join(OUT_DIR, 'main_results_table.csv'))
for l in SCHEMES.values():
    for s5f, s in [('audio-only (MuQ)', 'audio-only (MuQ)'), ('lyrics-only (SBERT)', 'lyrics-only (SBERT)'),
                   ('audio-only (MERT)', 'audio-only (MERT)'), ('audio-only (CLAP)', 'audio-only (CLAP)')]:
        r = ref[(ref.split == l) & (ref.strategy == s5f)].iloc[0]
        assert abs(r['P@10'] - RES_UNF[l][s]['P@K'][:, 9].mean()) < 1e-9, (l, s)
    late5f = ref[(ref.split == l) & ref.strategy.str.startswith('late fusion (')].iloc[0]
    assert abs(late5f['P@10'] - RES_UNF[l]['late fusion MuQ+SBERT']['P@K'][:, 9].mean()) < 1e-9
print('Unfiltered results match Phase 5f.')

def cluster_boot_ci(x, groups, n=N_BOOT):
    # resample artists (clusters of queries) with replacement
    codes, uniq = pd.factorize(groups); G = len(uniq)
    sums = np.bincount(codes, weights=x, minlength=G); cnts = np.bincount(codes, minlength=G)
    draw = rng.integers(0, G, (n, G))
    return np.percentile(sums[draw].sum(1) / cnts[draw].sum(1), [2.5, 97.5])

rows = []
for cond, RES in [('unfiltered', RES_UNF), ('artist-filtered', RES_ART), ('leader-filtered', RES_LEAD),
                  ('artist-filtered, re-selected', RES_RESEL)]:
    for l, res in RES.items():
        for s, m in res.items():
            r = {'condition': cond, 'split': l, 'strategy': s, 'n': len(m['nDCG@10'])}
            for col, x in [('P@10', m['P@K'][:, 9]), ('nDCG@10', m['nDCG@10']), ('AVdist@10', m['AVdist@10'])]:
                lo, hi = boot_ci(x); clo, chi = cluster_boot_ci(x, art[m['rows']])
                r.update({col: x.mean(), f'{col}_lo': lo, f'{col}_hi': hi, f'{col}_clo': clo, f'{col}_chi': chi})
            rows.append(r)
table = pd.DataFrame(rows)
table.to_csv(os.path.join(OUT_DIR, 'artist_filtered_main_table.csv'), index=False)

piv = table.pivot_table(index=['split', 'strategy'], columns='condition', values='P@10', sort=False)
print('P@10'); print(piv[['unfiltered', 'artist-filtered', 'leader-filtered', 'artist-filtered, re-selected']].round(3).to_string())
piv = table.pivot_table(index=['split', 'strategy'], columns='condition', values='nDCG@10', sort=False)
print('\nnDCG@10'); print(piv[['unfiltered', 'artist-filtered', 'leader-filtered', 'artist-filtered, re-selected']].round(3).to_string())
f = lambda r, c: f"{r[c]:.3f} [{r[c+'_lo']:.3f}, {r[c+'_hi']:.3f}] / cluster [{r[c+'_clo']:.3f}, {r[c+'_chi']:.3f}]"
print('\nArtist-filtered, with query-bootstrap and artist-cluster-bootstrap 95% CIs:')
t = table[table.condition == 'artist-filtered']
print(pd.DataFrame({c: t.apply(lambda r: f(r, c), axis=1).values for c in ['P@10', 'nDCG@10']},
                   index=(t.split + '  ' + t.strategy).values).to_string())

Unfiltered results match Phase 5f.


P@10
condition                        unfiltered  artist-filtered  leader-filtered  artist-filtered, re-selected
split    strategy                                                                                          
70/15/15 audio-only (MuQ)             0.625            0.610            0.610                         0.610
         audio-only (MERT)            0.618            0.602            0.601                         0.602
         lyrics-only (SBERT)          0.605            0.602            0.602                         0.602
         late fusion MuQ+SBERT        0.702            0.690            0.689                         0.690
         early fusion (≡ α=0.5)       0.704            0.694            0.694                         0.694
         audio-only (CLAP)            0.383            0.380            0.380                         0.380
         late fusion CLAP+SBERT       0.626            0.625            0.624                         0.619
40/30/30 audio-only (Mu

## 3. Paired change from filtering (filtered − unfiltered), per strategy

In [6]:
drows = []
for l in SCHEMES.values():
    for s in CFG_UNF[l]:
        for metric, get in [('P@10', lambda m: m['P@K'][:, 9]), ('nDCG@10', lambda m: m['nDCG@10'])]:
            d = get(RES_ART[l][s]) - get(RES_UNF[l][s]); lo, hi = boot_ci(d)
            drows.append({'split': l, 'strategy': s, 'metric': metric, 'delta': d.mean(), 'lo': lo, 'hi': hi})
delta = pd.DataFrame(drows)
print(delta.to_string(index=False, float_format=lambda x: f'{x:+.3f}'))
delta.to_csv(os.path.join(OUT_DIR, 'artist_filtered_delta.csv'), index=False)

# does filtering hurt audio more than lyrics?  difference-in-differences, paired over queries
for l in SCHEMES.values():
    for a in ['audio-only (MuQ)', 'audio-only (MERT)']:
        dd = ((RES_ART[l][a]['P@K'][:, 9] - RES_UNF[l][a]['P@K'][:, 9])
              - (RES_ART[l]['lyrics-only (SBERT)']['P@K'][:, 9] - RES_UNF[l]['lyrics-only (SBERT)']['P@K'][:, 9]))
        lo, hi = boot_ci(dd)
        print(f'{l}  Δ{a} − ΔSBERT (P@10): {dd.mean():+.3f} [{lo:+.3f}, {hi:+.3f}]')

   split               strategy  metric  delta     lo     hi
70/15/15       audio-only (MuQ)    P@10 -0.016 -0.022 -0.010
70/15/15       audio-only (MuQ) nDCG@10 -0.023 -0.031 -0.016
70/15/15      audio-only (MERT)    P@10 -0.017 -0.023 -0.011
70/15/15      audio-only (MERT) nDCG@10 -0.022 -0.028 -0.016
70/15/15    lyrics-only (SBERT)    P@10 -0.003 -0.005 -0.001
70/15/15    lyrics-only (SBERT) nDCG@10 -0.007 -0.010 -0.004
70/15/15  late fusion MuQ+SBERT    P@10 -0.012 -0.018 -0.007
70/15/15  late fusion MuQ+SBERT nDCG@10 -0.019 -0.025 -0.013
70/15/15 early fusion (≡ α=0.5)    P@10 -0.010 -0.015 -0.006
70/15/15 early fusion (≡ α=0.5) nDCG@10 -0.015 -0.021 -0.010
70/15/15      audio-only (CLAP)    P@10 -0.004 -0.006 -0.001
70/15/15      audio-only (CLAP) nDCG@10 -0.004 -0.007 -0.002
70/15/15 late fusion CLAP+SBERT    P@10 -0.002 -0.005 +0.001
70/15/15 late fusion CLAP+SBERT nDCG@10 -0.007 -0.011 -0.003
40/30/30       audio-only (MuQ)    P@10 -0.017 -0.021 -0.012
40/30/30       audio-onl

## 4. Paired tests under artist filtering (same comparisons and Holm families as Phase 5f)

In [7]:
def holm(p):
    p = np.asarray(p); order = np.argsort(p); m = len(p); adj = np.empty(m); run = 0.0
    for rank, i in enumerate(order):
        run = max(run, (m - rank) * p[i]); adj[i] = min(1.0, run)
    return adj

LATE, LATE_C = 'late fusion MuQ+SBERT', 'late fusion CLAP+SBERT'
PAIRS = [(LATE, 'lyrics-only (SBERT)'), (LATE, 'audio-only (MuQ)'), ('early fusion (≡ α=0.5)', 'lyrics-only (SBERT)'),
         ('audio-only (MuQ)', 'lyrics-only (SBERT)'), ('audio-only (MuQ)', 'audio-only (CLAP)'), (LATE, LATE_C)]
trows = []
for l, res in RES_ART.items():
    for metric, get in [('P@10', lambda m: m['P@K'][:, 9]), ('nDCG@10', lambda m: m['nDCG@10'])]:
        block = []
        for a, b in PAIRS:
            d = get(res[a]) - get(res[b]); lo, hi = boot_ci(d); clo, chi = cluster_boot_ci(d, art[res[a]['rows']])
            p = wilcoxon(get(res[a]), get(res[b]), zero_method='pratt').pvalue if (d != 0).any() else 1.0
            block.append({'split': l, 'metric': metric, 'comparison': f'{a} − {b}', 'diff': d.mean(),
                          'lo': lo, 'hi': hi, 'cluster_lo': clo, 'cluster_hi': chi, 'p': p})
        for r, ph in zip(block, holm([r['p'] for r in block])):
            r['p_holm'] = ph
        trows += block
tests = pd.DataFrame(trows)
print(tests.to_string(index=False, float_format=lambda x: f'{x:.3g}'))
tests.to_csv(os.path.join(OUT_DIR, 'artist_filtered_tests.csv'), index=False)

   split  metric                                     comparison     diff      lo     hi  cluster_lo  cluster_hi        p   p_holm
70/15/15    P@10    late fusion MuQ+SBERT − lyrics-only (SBERT)   0.0877  0.0589  0.118      0.0584       0.118 7.88e-08 2.36e-07
70/15/15    P@10       late fusion MuQ+SBERT − audio-only (MuQ)   0.0798   0.057  0.102      0.0568       0.104 2.28e-11 1.14e-10
70/15/15    P@10   early fusion (≡ α=0.5) − lyrics-only (SBERT)   0.0921  0.0656  0.118       0.066       0.119 1.54e-10 6.16e-10
70/15/15    P@10         audio-only (MuQ) − lyrics-only (SBERT)  0.00795 -0.0288 0.0447     -0.0289       0.047    0.916    0.916
70/15/15    P@10           audio-only (MuQ) − audio-only (CLAP)     0.23   0.193  0.267       0.191        0.27 3.73e-24 2.24e-23
70/15/15    P@10 late fusion MuQ+SBERT − late fusion CLAP+SBERT   0.0652  0.0387 0.0917      0.0375      0.0936 1.14e-05 2.27e-05
70/15/15 nDCG@10    late fusion MuQ+SBERT − lyrics-only (SBERT)   0.0514  0.0319 0.0707   

## 5. Equivalence (TOST) of audio vs lyrics under artist filtering

In [8]:
MARGIN = {'P@10': 0.05, 'nDCG@10': 0.03}
erows = []
for l, res in RES_ART.items():
    for audio in ['audio-only (MuQ)', 'audio-only (MERT)']:
        for metric, get in [('P@10', lambda m: m['P@K'][:, 9]), ('nDCG@10', lambda m: m['nDCG@10'])]:
            d = get(res[audio]) - get(res['lyrics-only (SBERT)']); n = len(d)
            bs = d[rng.integers(0, n, (N_BOOT, n))].mean(1); lo90, hi90 = np.percentile(bs, [5, 95])
            se = d.std(ddof=1) / np.sqrt(n); m = MARGIN[metric]
            p_tost = max(t_dist.sf((d.mean() + m) / se, n - 1), t_dist.cdf((d.mean() - m) / se, n - 1))
            erows.append({'split': l, 'audio': audio, 'metric': metric, 'mean_diff (audio − lyrics)': d.mean(),
                          'ci90_lo': lo90, 'ci90_hi': hi90, 'margin': m, 'equivalent': (lo90 > -m) and (hi90 < m),
                          'p_TOST': p_tost})
equiv = pd.DataFrame(erows)
print(equiv.to_string(index=False, float_format=lambda x: f'{x:.4g}'))
equiv.to_csv(os.path.join(OUT_DIR, 'artist_filtered_equivalence.csv'), index=False)

   split             audio  metric  mean_diff (audio − lyrics)  ci90_lo  ci90_hi  margin  equivalent    p_TOST
70/15/15  audio-only (MuQ)    P@10                    0.007947 -0.02252  0.03874    0.05        True   0.01401
70/15/15  audio-only (MuQ) nDCG@10                   -0.002918 -0.02431  0.01856    0.03        True   0.01851
70/15/15 audio-only (MERT)    P@10                  -0.0003311 -0.03311  0.03245    0.05        True  0.006544
70/15/15 audio-only (MERT) nDCG@10                   -0.008385 -0.03055  0.01401    0.03       False   0.05604
40/30/30  audio-only (MuQ)    P@10                    0.006483 -0.01475  0.02788    0.05        True 0.0004538
40/30/30  audio-only (MuQ) nDCG@10                   0.0001214 -0.01509  0.01533    0.03        True 0.0006957
40/30/30 audio-only (MERT)    P@10                   0.0003241 -0.02042  0.02107    0.05        True 6.216e-05
40/30/30 audio-only (MERT) nDCG@10                    -0.01076 -0.02555 0.004006    0.03        True   0.01502


## 6. Per-quadrant P@10, unfiltered vs artist-filtered

In [9]:
qrows = []
for cond, RES in [('unfiltered', RES_UNF), ('artist-filtered', RES_ART)]:
    for l, res in RES.items():
        for s in ['audio-only (MuQ)', 'lyrics-only (SBERT)', LATE]:
            m = res[s]
            for q in QUADS:
                v = m['P@K'][m['quadrant'] == q, 9]; lo, hi = boot_ci(v, 2000)
                qrows.append({'condition': cond, 'split': l, 'strategy': s, 'quadrant': q, 'n': len(v),
                              'P@10': v.mean(), 'lo': lo, 'hi': hi})
pq = pd.DataFrame(qrows)
pq.to_csv(os.path.join(OUT_DIR, 'artist_filtered_per_quadrant.csv'), index=False)
print(pq.pivot_table(index=['split', 'strategy', 'condition'], columns='quadrant', values='P@10', sort=False).round(3).to_string())

# chance per quadrant after filtering (query excluded, same-artist excluded), averaged over test queries
for l in SCHEMES.values():
    rows_ = RES_ART[l][LATE]['rows']
    ch = [(quad[~SAME['artist'][i] & (np.arange(len(quad)) != i)] == quad[i]).mean() for i in rows_]
    print(l, 'filtered chance by quadrant:', pd.Series(ch).groupby(quad[rows_]).mean().round(3).to_dict(),
          ' overall', round(float(np.mean(ch)), 3))
print('Saved: artist_filtered_config_selection.csv, artist_overlap_top10.csv, artist_filtered_main_table.csv, '
      'artist_filtered_delta.csv, artist_filtered_tests.csv, artist_filtered_equivalence.csv, artist_filtered_per_quadrant.csv')

quadrant                                           Q1     Q2     Q3     Q4
split    strategy              condition                                  
70/15/15 audio-only (MuQ)      unfiltered       0.613  0.875  0.505  0.512
                               artist-filtered  0.600  0.857  0.483  0.503
         lyrics-only (SBERT)   unfiltered       0.483  0.873  0.457  0.609
                               artist-filtered  0.476  0.872  0.455  0.609
         late fusion MuQ+SBERT unfiltered       0.636  0.943  0.557  0.676
                               artist-filtered  0.613  0.939  0.540  0.671
40/30/30 audio-only (MuQ)      unfiltered       0.653  0.870  0.520  0.492
                               artist-filtered  0.635  0.858  0.495  0.478
         lyrics-only (SBERT)   unfiltered       0.526  0.854  0.463  0.614
                               artist-filtered  0.520  0.849  0.458  0.613
         late fusion MuQ+SBERT unfiltered       0.681  0.927  0.563  0.656
                         